# Streaming ETL and country-based routing

This walkthrough documents the ETL stage. A Python consumer reads the source Kinesis stream, enriches each session event, and sends it to either the USA or International stream. Firehose then delivers each stream to its corresponding S3 bucket.

## 1. Transform a session event

The pipeline adds a timestamp and three derived metrics.

In [ ]:
import copy
import datetime
import json
from pathlib import Path

record = json.loads(Path("data/example_record.json").read_text())

def transform_session(session):
    enriched = copy.deepcopy(session)
    enriched["processing_timestamp"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    products = enriched["browse_history"]
    enriched["overall_product_quantity"] = sum(int(p["quantity"]) for p in products)
    enriched["overall_in_shopping_cart"] = sum(int(p["quantity"]) for p in products if p["in_shopping_cart"] is True)
    enriched["total_different_products"] = len(products)
    return enriched

enriched = transform_session(record)
enriched

## 2. Route by country

Records with `country == "USA"` go to the USA stream. Every other record goes to the International stream.

In [ ]:
destination_streams = {
    "USA": "YOUR_USA_DESTINATION_STREAM",
    "International": "YOUR_INTERNATIONAL_DESTINATION_STREAM",
}
destination = destination_streams["USA"] if enriched["country"] == "USA" else destination_streams["International"]
print(destination)

## 3. Understand the delivery layer

Each destination stream is connected to an Amazon Data Firehose delivery stream configured to write to its matching S3 bucket. This notebook does not create or modify AWS resources.

## Production considerations

The lab uses in-memory shard iterators and simple logging. A real system should add durable checkpoints, retries with backoff, idempotency, schema validation, metrics, alarms, and dead-letter handling. Also note that `len(browse_history)` counts entries, not unique product codes.